In [1]:
import pickle

PREPROCESSED_PATH = "/media/user/DataDisk/Python/Dordrecht/AI Model/GNN-model/gnn_preprocessed.pkl"

with open(PREPROCESSED_PATH, "rb") as f:
    pre = pickle.load(f)

# Unpack everything
data_filled  = pre["data_filled"]
flagged      = pre["flagged"]
dates        = pre["dates"]
stations     = pre["stations"]
feature_cols = pre["feature_cols"]
edge_index   = pre["edge_index"]
edge_attr    = pre["edge_attr"]
scalers      = pre["scalers"]
rh_idx       = pre["rh_idx"]
config       = pre["config"]

T, N, F = data_filled.shape

print("Loaded successfully")
print(f"  data_filled shape : {data_filled.shape}")
print(f"  flagged days      : {flagged.sum():,} / {T:,}")
print(f"  config            : {config}")

Loaded successfully
  data_filled shape : (10897, 34, 46)
  flagged days      : 2,151 / 10,897
  config            : {'window_size': 7, 'skip_threshold': 0.3, 'min_bad_stations': 3, 'core_features': ['TG', 'TN', 'TX', 'UG', 'RH', 'FG', 'Q']}


In [2]:
import torch

# We build sample i=100 manually so we can see exactly what shape the model receives
i           = 100
WINDOW_SIZE = config["window_size"]   # 7

start = i
end   = i + WINDOW_SIZE   # target day index

print(f"Input window : day {start} to day {end-1}")
print(f"Input dates  : {dates[start].date()} → {dates[end-1].date()}")
print(f"Target day   : day {end}  ({dates[end].date()})")

# Slice the input window — raw shape [W, N, F] = [7, 34, 46]
window = data_filled[start:end]
print(f"\nwindow shape (raw) : {window.shape}")

Input window : day 100 to day 106
Input dates  : 1995-04-11 → 1995-04-17
Target day   : day 107  (1995-04-18)

window shape (raw) : (7, 34, 46)


In [3]:
# Transpose from [W, N, F] to [N, W, F]
# The GRU will iterate over the W (time) axis per node
# numpy transpose(1, 0, 2) means:
#   new axis 0 ← old axis 1 (N=stations)
#   new axis 1 ← old axis 0 (W=days)
#   new axis 2 ← old axis 2 (F=features)
window_NWF = window.transpose(1, 0, 2)
print(f"window shape after transpose : {window_NWF.shape}")  # (34, 7, 46)

# Convert to torch tensor
x = torch.tensor(window_NWF, dtype=torch.float32)
print(f"x shape (torch)              : {tuple(x.shape)}")    # (34, 7, 46)

window shape after transpose : (34, 7, 46)
x shape (torch)              : (34, 7, 46)


In [4]:
# Take RH (index 13) at the target day for all 34 stations
# data_filled[end] shape = [N, F] — we take only the RH column
y_raw = data_filled[end, :, rh_idx]   # shape [N] = [34]
print(f"y_raw shape : {y_raw.shape}")

# Unsqueeze to [N, 1] — consistent for the model output layer
y = torch.tensor(y_raw, dtype=torch.float32).unsqueeze(1)
print(f"y shape     : {tuple(y.shape)}")   # (34, 1)

# y is still normalised — let's see what the actual mm values look like
rh_scaler = scalers[rh_idx]
y_mm = rh_scaler.inverse_transform(y.numpy())   # [N, 1] in mm

print(f"\nNormalised values (first 5 stations) : {y[:5, 0].tolist()}")
print(f"Millimetre values (first 5 stations) : {y_mm[:5, 0].round(2).tolist()}")

y_raw shape : (34,)
y shape     : (34, 1)

Normalised values (first 5 stations) : [0.37055426836013794, 1.6267287731170654, 0.0, 0.5468593835830688, 0.6790884137153625]
Millimetre values (first 5 stations) : [3.9000000953674316, 9.600000381469727, 2.2200000286102295, 4.699999809265137, 5.300000190734863]


In [5]:
from torch_geometric.data import Data

# Convert graph arrays to torch tensors
ei = torch.tensor(edge_index, dtype=torch.long)     # [2, 170]
ea = torch.tensor(edge_attr,  dtype=torch.float32)  # [170]

# Pack everything into one PyG Data object
sample = Data(
    x          = x,    # [34, 7, 46]  — input window
    y          = y,    # [34, 1]      — target RH
    edge_index = ei,   # [2, 170]     — graph edges
    edge_attr  = ea,   # [170]        — edge weights
    pred_date  = str(dates[end].date()),   # "1995-04-18"
)

print(sample)
print(f"\nsample.x.shape         : {tuple(sample.x.shape)}")
print(f"sample.y.shape         : {tuple(sample.y.shape)}")
print(f"sample.edge_index.shape: {tuple(sample.edge_index.shape)}")
print(f"sample.pred_date       : {sample.pred_date}")

Data(x=[34, 7, 46], edge_index=[2, 170], edge_attr=[170], y=[34, 1], pred_date='1995-04-18')

sample.x.shape         : (34, 7, 46)
sample.y.shape         : (34, 1)
sample.edge_index.shape: (2, 170)
sample.pred_date       : 1995-04-18


In [6]:
from torch.utils.data import Dataset
from torch_geometric.data import Data
import torch
import numpy as np

class RainfallWindowDataset(Dataset):

    def __init__(self, data_filled, flagged, dates, edge_index, edge_attr, rh_idx, window_size=7):
        self.data        = data_filled
        self.flagged     = flagged
        self.dates       = dates
        self.rh_idx      = rh_idx
        self.window_size = window_size

        # Convert graph to torch once — same for every sample
        self.edge_index = torch.tensor(edge_index, dtype=torch.long)
        self.edge_attr  = torch.tensor(edge_attr,  dtype=torch.float32)

        T = len(data_filled)

        # Build list of valid start indices
        # A sample is valid if none of the days in [start, end] are flagged
        self.valid_indices = []
        for i in range(T - window_size):
            end = i + window_size
            if not flagged[i : end + 1].any():
                self.valid_indices.append(i)

        print(f"Total samples    : {T - window_size:,}")
        print(f"Valid samples    : {len(self.valid_indices):,}")
        print(f"Skipped (flagged): {T - window_size - len(self.valid_indices):,}")

    def __len__(self):
        return len(self.valid_indices)

    def __getitem__(self, idx):
        start = self.valid_indices[idx]
        end   = start + self.window_size

        # Input window [N, W, F]
        x = torch.tensor(
            self.data[start:end].transpose(1, 0, 2),
            dtype=torch.float32
        )

        # Target RH [N, 1]
        y = torch.tensor(
            self.data[end, :, self.rh_idx],
            dtype=torch.float32
        ).unsqueeze(1)

        return Data(
            x          = x,
            y          = y,
            edge_index = self.edge_index,
            edge_attr  = self.edge_attr,
            pred_date  = str(self.dates[end].date()),
        )


# Test it
dataset = RainfallWindowDataset(
    data_filled = data_filled,
    flagged     = flagged,
    dates       = dates,
    edge_index  = edge_index,
    edge_attr   = edge_attr,
    rh_idx      = rh_idx,
    window_size = config["window_size"],
)

# Check a sample
sample = dataset[100]
print(f"\ndataset[100].x.shape     : {tuple(sample.x.shape)}")
print(f"dataset[100].y.shape     : {tuple(sample.y.shape)}")
print(f"dataset[100].pred_date   : {sample.pred_date}")

Total samples    : 10,890
Valid samples    : 8,700
Skipped (flagged): 2,190

dataset[100].x.shape     : (34, 7, 46)
dataset[100].y.shape     : (34, 1)
dataset[100].pred_date   : 1999-07-09


In [7]:
from torch.utils.data import Subset

n = len(dataset)   # 8,700

# Compute split boundaries by index position
train_end = int(n * 0.70)
val_end   = int(n * 0.85)

train_indices = list(range(0,         train_end))
val_indices   = list(range(train_end, val_end))
test_indices  = list(range(val_end,   n))

print(f"Total samples : {n:,}")
print(f"Train         : {len(train_indices):,}  ({len(train_indices)/n:.0%})")
print(f"Val           : {len(val_indices):,}   ({len(val_indices)/n:.0%})")
print(f"Test          : {len(test_indices):,}   ({len(test_indices)/n:.0%})")

# Check the actual date boundaries
print(f"\nTrain : {dataset[train_indices[0]].pred_date}  →  {dataset[train_indices[-1]].pred_date}")
print(f"Val   : {dataset[val_indices[0]].pred_date}  →  {dataset[val_indices[-1]].pred_date}")
print(f"Test  : {dataset[test_indices[0]].pred_date}  →  {dataset[test_indices[-1]].pred_date}")

# Wrap with Subset — no data copied, just index views
train_set = Subset(dataset, train_indices)
val_set   = Subset(dataset, val_indices)
test_set  = Subset(dataset, test_indices)

Total samples : 8,700
Train         : 6,090  (70%)
Val           : 1,305   (15%)
Test          : 1,305   (15%)

Train : 1999-03-31  →  2015-12-01
Val   : 2015-12-02  →  2021-02-15
Test  : 2021-02-16  →  2024-10-31


In [8]:
from torch_geometric.loader import DataLoader

train_loader = DataLoader(train_set, batch_size=32, shuffle=True)
val_loader   = DataLoader(val_set,   batch_size=32, shuffle=False)
test_loader  = DataLoader(test_set,  batch_size=32, shuffle=False)

print(f"Train batches : {len(train_loader)}")
print(f"Val   batches : {len(val_loader)}")
print(f"Test  batches : {len(test_loader)}")

# Inspect one batch
batch = next(iter(train_loader))
print(f"\nbatch.x.shape          : {tuple(batch.x.shape)}")
print(f"batch.y.shape          : {tuple(batch.y.shape)}")
print(f"batch.edge_index.shape : {tuple(batch.edge_index.shape)}")
print(f"batch.batch.shape      : {tuple(batch.batch.shape)}")
print(f"Unique graphs in batch : {batch.batch.unique().numel()}")

Train batches : 191
Val   batches : 41
Test  batches : 41

batch.x.shape          : (1088, 7, 46)
batch.y.shape          : (1088, 1)
batch.edge_index.shape : (2, 5440)
batch.batch.shape      : (1088,)
Unique graphs in batch : 32


In [9]:
split = {
    "train_indices" : train_indices,
    "val_indices"   : val_indices,
    "test_indices"  : test_indices,
}

SPLIT_PATH = "/media/user/DataDisk/Python/Dordrecht/AI Model/GNN-model/gnn_split.pkl"

with open(SPLIT_PATH, "wb") as f:
    pickle.dump(split, f)

print(f"Saved to {SPLIT_PATH}")

Saved to /media/user/DataDisk/Python/Dordrecht/AI Model/GNN-model/gnn_split.pkl


Now Let's start to build the model

In [ ]:
# 1
import torch
import torch.nn as nn

# Take one batch from the dataloader
batch = next(iter(train_loader))

# batch.x shape : [1088, 7, 46]
# That is        [N_total, W, F]
x = batch.x
print(f"Input shape : {tuple(x.shape)}")

# Define a simple GRU
# input_size  = 46  (one feature vector per day)
# hidden_size = 64  (size of the hidden state we pass to GAT)
# batch_first = True means input shape is [batch, seq, features]
#               which matches our [N_total, W, F]
gru = nn.GRU(input_size=46, hidden_size=64, batch_first=True)

# Run the GRU
# output shape : [N_total, W, 64]  — hidden state at every timestep
# h_n shape    : [1, N_total, 64]  — hidden state at the LAST timestep only
output, h_n = gru(x)

print(f"GRU output shape : {tuple(output.shape)}")
print(f"GRU h_n shape    : {tuple(h_n.shape)}")

# We only need the last hidden state — squeeze out the first dimension
h = h_n.squeeze(0)   # [N_total, 64]
print(f"h shape          : {tuple(h.shape)}")

Input shape : (1088, 7, 46)
GRU output shape : (1088, 7, 64)
GRU h_n shape    : (1, 1088, 64)
h shape          : (1088, 64)


In [ ]:
# 2 
from torch_geometric.nn import GATConv

# Define one GAT layer
# in_channels  = 64  (what comes out of the GRU)
# out_channels = 32  (size per attention head)
# heads        = 4   (4 independent attention mechanisms)
# concat=True means outputs are concatenated → final size = 32 × 4 = 128
gat = GATConv(in_channels=64, out_channels=32, heads=4, concat=True)

# Run the GAT
# h            : [N_total, 64]   — node features from GRU
# batch.edge_index : [2, 5440]   — which stations are connected
out = gat(h, batch.edge_index)

print(f"GAT input shape  : {tuple(h.shape)}")
print(f"GAT output shape : {tuple(out.shape)}")
# expect [1088, 128]  because 32 × 4 heads = 128

GAT input shape  : (1088, 64)
GAT output shape : (1088, 128)


In [ ]:
# 3 
# Define a simple linear layer
# in_features  = 128  (what comes out of GAT)
# out_features = 1    (predicted RH for tomorrow)
linear = nn.Linear(in_features=128, out_features=1)

# Run the linear layer
pred = linear(out)

print(f"Linear input shape  : {tuple(out.shape)}")
print(f"Linear output shape : {tuple(pred.shape)}")
# expect [1088, 1] — one prediction per station

Linear input shape  : (1088, 128)
Linear output shape : (1088, 1)


Now all three parts together

In [13]:
import torch
import torch.nn as nn
from torch_geometric.nn import GATConv

class RainfallGNN(nn.Module):

    def __init__(self, n_features, hidden_dim, gat_heads, gat_out_dim, dropout=0.2):
        super().__init__()

        # Part 1 — GRU
        # Reads 7-day history per station
        # Input  : [N, 7, n_features]
        # Output : [N, hidden_dim]
        self.gru = nn.GRU(
            input_size  = n_features,
            hidden_size = hidden_dim,
            batch_first = True,
        )

        # Part 2 — GAT
        # Stations share information with neighbours
        # Input  : [N, hidden_dim]
        # Output : [N, gat_out_dim * gat_heads]
        self.gat = GATConv(
            in_channels  = hidden_dim,
            out_channels = gat_out_dim,
            heads        = gat_heads,
            concat       = True,
        )

        # Part 3 — Linear head
        # One prediction per station
        # Input  : [N, gat_out_dim * gat_heads]
        # Output : [N, 1]
        self.linear = nn.Linear(gat_out_dim * gat_heads, 1)

        self.dropout = nn.Dropout(dropout)

    def forward(self, x, edge_index):
        # ── GRU ───────────────────────────────────────────────────────────────
        _, h_n = self.gru(x)       # h_n : [1, N, hidden_dim]
        h = h_n.squeeze(0)         # h   : [N, hidden_dim]
        h = self.dropout(h)

        # ── GAT ───────────────────────────────────────────────────────────────
        h = self.gat(h, edge_index)  # h : [N, gat_out_dim * heads]
        h = torch.relu(h)
        h = self.dropout(h)

        # ── Linear ────────────────────────────────────────────────────────────
        out = self.linear(h)         # out : [N, 1]
        return out


# Instantiate the model
model = RainfallGNN(
    n_features  = 46,   # F
    hidden_dim  = 64,   # GRU hidden size
    gat_heads   = 4,    # attention heads
    gat_out_dim = 32,   # per-head output size → 32×4=128 total
    dropout     = 0.2,
)

print(model)

# Count trainable parameters
n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"\nTrainable parameters : {n_params:,}")

# Test one forward pass
batch = next(iter(train_loader))
pred  = model(batch.x, batch.edge_index)
print(f"\nInput shape  : {tuple(batch.x.shape)}")
print(f"Output shape : {tuple(pred.shape)}")   # expect [1088, 1]

RainfallGNN(
  (gru): GRU(46, 64, batch_first=True)
  (gat): GATConv(64, 32, heads=4)
  (linear): Linear(in_features=128, out_features=1, bias=True)
  (dropout): Dropout(p=0.2, inplace=False)
)

Trainable parameters : 30,209

Input shape  : (1088, 7, 46)
Output shape : (1088, 1)


In [14]:
import torch.optim as optim

# Loss function: MSE (Mean Squared Error)
# We predict normalised RH values, MSE penalises large errors more
criterion = nn.MSELoss()

# Optimizer: Adam with learning rate 0.001
optimizer = optim.Adam(model.parameters(), lr=0.001)

print(f"Loss function : {criterion}")
print(f"Optimizer     : {optimizer.__class__.__name__}")

# Quick test — compute loss between prediction and target
batch = next(iter(train_loader))
pred  = model(batch.x, batch.edge_index)   # [1088, 1]
loss  = criterion(pred, batch.y)           # compare to [1088, 1]

print(f"\nPred shape  : {tuple(pred.shape)}")
print(f"Target shape: {tuple(batch.y.shape)}")
print(f"Loss value  : {loss.item():.4f}")

Loss function : MSELoss()
Optimizer     : Adam

Pred shape  : (1088, 1)
Target shape: (1088, 1)
Loss value  : 0.8286


In [15]:
# A single training step does 4 things:
# 1. Forward pass  — model makes predictions
# 2. Loss          — how wrong were the predictions
# 3. Backward pass — compute gradients
# 4. Optimizer step — update weights

model.train()   # tells dropout to be active during training

# 1. Forward pass
batch = next(iter(train_loader))
pred  = model(batch.x, batch.edge_index)   # [1088, 1]

# 2. Compute loss
loss = criterion(pred, batch.y)

# 3. Clear old gradients, then compute new ones
optimizer.zero_grad()
loss.backward()

# 4. Update weights
optimizer.step()

print(f"Loss after one step : {loss.item():.4f}")

# Run a second step to confirm loss is actually changing
pred2 = model(batch.x, batch.edge_index)
loss2 = criterion(pred2, batch.y)
print(f"Loss after two steps: {loss2.item():.4f}")
print(f"Loss decreased      : {loss2.item() < loss.item()}")

Loss after one step : 1.0027
Loss after two steps: 0.9718
Loss decreased      : True


In [16]:
def train_one_epoch(model, loader, criterion, optimizer):
    model.train()
    
    total_loss = 0
    total_mae  = 0
    n_batches  = len(loader)

    for batch in loader:
        # Forward pass
        pred = model(batch.x, batch.edge_index)

        # Loss
        loss = criterion(pred, batch.y)

        # Backward pass
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        # Track loss and MAE
        total_loss += loss.item()
        total_mae  += torch.abs(pred - batch.y).mean().item()

    avg_loss = total_loss / n_batches
    avg_mae  = total_mae  / n_batches

    return avg_loss, avg_mae


# Run one epoch and time it
import time
t0 = time.time()

train_loss, train_mae = train_one_epoch(model, train_loader, criterion, optimizer)

print(f"Epoch time : {time.time() - t0:.1f}s")
print(f"Train loss : {train_loss:.4f}")
print(f"Train MAE  : {train_mae:.4f}")

Epoch time : 189.7s
Train loss : 0.8617
Train MAE  : 0.5442


In [19]:
# Revert to 0 workers — simpler and same speed on CPU
train_loader = DataLoader(train_set, batch_size=32, shuffle=True,  num_workers=0)
val_loader   = DataLoader(val_set,   batch_size=32, shuffle=False, num_workers=0)
test_loader  = DataLoader(test_set,  batch_size=32, shuffle=False, num_workers=0)

print("Loaders ready")
print(f"Estimated time for 20 epochs : {20 * 190 / 60:.0f} mins")
print(f"Estimated time for 50 epochs : {50 * 190 / 60:.0f} mins")

Loaders ready
Estimated time for 20 epochs : 63 mins
Estimated time for 50 epochs : 158 mins


In [20]:
def validate(model, loader, criterion):
    model.eval()   # turns off dropout during validation
    
    total_loss = 0
    total_mae  = 0
    n_batches  = len(loader)

    with torch.no_grad():   # no gradients needed — saves memory and time
        for batch in loader:
            pred = model(batch.x, batch.edge_index)
            loss = criterion(pred, batch.y)

            total_loss += loss.item()
            total_mae  += torch.abs(pred - batch.y).mean().item()

    avg_loss = total_loss / n_batches
    avg_mae  = total_mae  / n_batches

    return avg_loss, avg_mae


# Quick test
val_loss, val_mae = validate(model, val_loader, criterion)
print(f"Val loss : {val_loss:.4f}")
print(f"Val MAE  : {val_mae:.4f}")

Val loss : 0.6686
Val MAE  : 0.4878


In [21]:
# Check if MLflow is installed
try:
    import mlflow
    print(f"MLflow version : {mlflow.__version__}")
except ImportError:
    print("MLflow not installed — run:  pip install mlflow")

MLflow not installed — run:  pip install mlflow


In [22]:
pip install mlflow

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 10.4 MB/s  0:00:006m0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 6.9 MB/s  0:00:000m eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 3.5 MB/s  0:00:000m eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.5/4.5 MB 8.6 MB/s  0:00:0036m0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 832.8/832.8 kB 19.8 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.4/12.4 MB 11.8 MB/s  0:00:016m0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 MB 62.2 MB/s  0:00:006m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 40.6 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 45.2 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 613.9/613.9 kB 2.4 MB/s  0:00:00
  Attempting uninstall: pandas╺━━━━━━━━━━━━━━━━━━ 25/47 [pyasn1-modules]]oto]
    Found existing installation: pandas 3.0.0━━━━━━━━━━━━━━━━━ 25/47 [pyasn1-modules]
    Uninstalling

In [1]:
import pickle
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, Subset
from torch_geometric.data import Data
from torch_geometric.loader import DataLoader
from torch_geometric.nn import GATConv
import mlflow
import time

# ── Reload preprocessed data ──────────────────────────────────────────────────
PREPROCESSED_PATH = "/media/user/DataDisk/Python/Dordrecht/AI Model/GNN-model/gnn_preprocessed.pkl"
SPLIT_PATH        = "/media/user/DataDisk/Python/Dordrecht/AI Model/GNN-model/gnn_split.pkl"

with open(PREPROCESSED_PATH, "rb") as f:
    pre = pickle.load(f)

with open(SPLIT_PATH, "rb") as f:
    split = pickle.load(f)

data_filled  = pre["data_filled"]
flagged      = pre["flagged"]
dates        = pre["dates"]
stations     = pre["stations"]
feature_cols = pre["feature_cols"]
edge_index   = pre["edge_index"]
edge_attr    = pre["edge_attr"]
scalers      = pre["scalers"]
rh_idx       = pre["rh_idx"]
config       = pre["config"]
T, N, F      = data_filled.shape

print("Data loaded ✅")
print(f"MLflow version : {mlflow.__version__} ✅")

Data loaded ✅
MLflow version : 3.10.1 ✅


Now rebuild the dataset, model and loaders in one cell:

In [2]:
# ── Dataset class ─────────────────────────────────────────────────────────────
class RainfallWindowDataset(Dataset):
    def __init__(self, data_filled, flagged, dates, edge_index, edge_attr, rh_idx, window_size=7):
        self.data        = data_filled
        self.flagged     = flagged
        self.dates       = dates
        self.rh_idx      = rh_idx
        self.window_size = window_size
        self.edge_index  = torch.tensor(edge_index, dtype=torch.long)
        self.edge_attr   = torch.tensor(edge_attr,  dtype=torch.float32)
        T = len(data_filled)
        self.valid_indices = [i for i in range(T - window_size)
                              if not flagged[i : i + window_size + 1].any()]
        print(f"Valid samples : {len(self.valid_indices):,}")

    def __len__(self):
        return len(self.valid_indices)

    def __getitem__(self, idx):
        start = self.valid_indices[idx]
        end   = start + self.window_size
        x = torch.tensor(self.data[start:end].transpose(1, 0, 2), dtype=torch.float32)
        y = torch.tensor(self.data[end, :, self.rh_idx], dtype=torch.float32).unsqueeze(1)
        return Data(x=x, y=y, edge_index=self.edge_index, edge_attr=self.edge_attr,
                    pred_date=str(self.dates[end].date()))

# ── Model class ───────────────────────────────────────────────────────────────
class RainfallGNN(nn.Module):
    def __init__(self, n_features, hidden_dim, gat_heads, gat_out_dim, dropout=0.2):
        super().__init__()
        self.gru    = nn.GRU(input_size=n_features, hidden_size=hidden_dim, batch_first=True)
        self.gat    = GATConv(in_channels=hidden_dim, out_channels=gat_out_dim, heads=gat_heads, concat=True)
        self.linear = nn.Linear(gat_out_dim * gat_heads, 1)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, edge_index):
        _, h_n = self.gru(x)
        h = h_n.squeeze(0)
        h = self.dropout(h)
        h = self.gat(h, edge_index)
        h = torch.relu(h)
        h = self.dropout(h)
        return self.linear(h)

# ── Build dataset, splits, loaders ───────────────────────────────────────────
dataset    = RainfallWindowDataset(data_filled, flagged, dates, edge_index, edge_attr, rh_idx)
train_set  = Subset(dataset, split["train_indices"])
val_set    = Subset(dataset, split["val_indices"])
test_set   = Subset(dataset, split["test_indices"])

train_loader = DataLoader(train_set, batch_size=32, shuffle=True)
val_loader   = DataLoader(val_set,   batch_size=32, shuffle=False)
test_loader  = DataLoader(test_set,  batch_size=32, shuffle=False)

# ── Model, loss, optimizer ────────────────────────────────────────────────────
model     = RainfallGNN(n_features=46, hidden_dim=64, gat_heads=4, gat_out_dim=32, dropout=0.2)
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

print("Model ready ✅")
print(f"Trainable parameters : {sum(p.numel() for p in model.parameters()):,}")

Valid samples : 8,700
Model ready ✅
Trainable parameters : 30,209


In [3]:
def train_one_epoch(model, loader, criterion, optimizer):
    model.train()
    total_loss = 0
    total_mae  = 0

    for batch in loader:
        pred = model(batch.x, batch.edge_index)
        loss = criterion(pred, batch.y)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
        total_mae  += torch.abs(pred - batch.y).mean().item()

    return total_loss / len(loader), total_mae / len(loader)


def validate(model, loader, criterion):
    model.eval()
    total_loss = 0
    total_mae  = 0

    with torch.no_grad():
        for batch in loader:
            pred = model(batch.x, batch.edge_index)
            loss = criterion(pred, batch.y)
            total_loss += loss.item()
            total_mae  += torch.abs(pred - batch.y).mean().item()

    return total_loss / len(loader), total_mae / len(loader)

print("Functions ready ✅")

Functions ready ✅


In [5]:
# ── MLflow training loop ──────────────────────────────────────────────────────
EPOCHS    = 50
BEST_VAL  = float("inf")
BEST_EPOCH = 0

# Set the experiment name — all runs go under this name in the dashboard
mlflow.set_experiment("rainfall_gnn")

with mlflow.start_run():

    # Log all hyperparameters once at the start
    mlflow.log_params({
        "window_size"  : config["window_size"],
        "n_features"   : F,
        "n_stations"   : N,
        "hidden_dim"   : 64,
        "gat_heads"    : 4,
        "gat_out_dim"  : 32,
        "dropout"      : 0.2,
        "learning_rate": 0.001,
        "batch_size"   : 32,
        "epochs"       : EPOCHS,
    })

    t_start = time.time()

    for epoch in range(1, EPOCHS + 1):
        t_epoch = time.time()

        # Train and validate
        train_loss, train_mae = train_one_epoch(model, train_loader, criterion, optimizer)
        val_loss,   val_mae   = validate(model, val_loader, criterion)

        epoch_time = time.time() - t_epoch

        # Log metrics to MLflow after every epoch
        mlflow.log_metrics({
            "train_loss" : train_loss,
            "train_mae"  : train_mae,
            "val_loss"   : val_loss,
            "val_mae"    : val_mae,
        }, step=epoch)

        # Save best model
        if val_loss < BEST_VAL:
            BEST_VAL   = val_loss
            BEST_EPOCH = epoch
            torch.save(model.state_dict(),
                "/media/user/DataDisk/Python/Dordrecht/AI Model/GNN-model/best_model.pt")

        # Print progress
        print(f"Epoch {epoch:3d}/{EPOCHS} | "
              f"train_loss {train_loss:.4f} | "
              f"val_loss {val_loss:.4f} | "
              f"val_mae {val_mae:.4f} | "
              f"time {epoch_time:.0f}s")

    total_time = time.time() - t_start

    # Log final summary metrics
    mlflow.log_metrics({
        "best_val_loss" : BEST_VAL,
        "best_epoch"    : BEST_EPOCH,
        "total_time_s"  : total_time,
    })

    print(f"\nDone — best val_loss {BEST_VAL:.4f} at epoch {BEST_EPOCH}")
    print(f"Total time : {total_time/60:.1f} mins")
    print(f"Best model saved ✅")

Epoch   1/50 | train_loss 0.6635 | val_loss 0.7480 | val_mae 0.5073 | time 179s
Epoch   2/50 | train_loss 0.6534 | val_loss 0.7331 | val_mae 0.5040 | time 201s
Epoch   3/50 | train_loss 0.6427 | val_loss 0.7157 | val_mae 0.4973 | time 129s
Epoch   4/50 | train_loss 0.6316 | val_loss 0.7356 | val_mae 0.5110 | time 124s
Epoch   5/50 | train_loss 0.6179 | val_loss 0.7369 | val_mae 0.4994 | time 29s
Epoch   6/50 | train_loss 0.6132 | val_loss 0.7222 | val_mae 0.4886 | time 24s
Epoch   7/50 | train_loss 0.5989 | val_loss 0.7414 | val_mae 0.4987 | time 26s
Epoch   8/50 | train_loss 0.5889 | val_loss 0.7414 | val_mae 0.5032 | time 24s
Epoch   9/50 | train_loss 0.5787 | val_loss 0.7300 | val_mae 0.4952 | time 23s
Epoch  10/50 | train_loss 0.5741 | val_loss 0.7480 | val_mae 0.4884 | time 25s
Epoch  11/50 | train_loss 0.5689 | val_loss 0.7493 | val_mae 0.5083 | time 25s
Epoch  12/50 | train_loss 0.5583 | val_loss 0.7422 | val_mae 0.4813 | time 26s
Epoch  13/50 | train_loss 0.5550 | val_loss 0.76